# HTTP

**HTTP** — это протокол прикладного уровня, по которому клиент (браузер, requests, curl) общается с сервером

**Структура HTTP-запроса:**
| Часть | Пример | Что означает | Обязательность |
|-------|--------|--------------|----------------|
| **Стартовая строка** | `GET /api/users/42?fields=name HTTP/1.1` | Метод + путь + query + версия | Обязательно |
| Заголовок | `Host: api.example.com` | К какому сайту обращаемся | Обязательно (HTTP/1.1) |
| Заголовок | `User-Agent: python-requests/2.31.0` | Кто клиент (браузер (Chrome, Firefox), скрипт (requests, curl), мобильное приложение) | Необязательно |
| Заголовок | `Accept: application/json` | Что хотим получить (говорит серверу, какой формат ответа хотим: JSON, HTML, XML, картинку) | Необязательно |
| Заголовок | `Authorization: Bearer abc123` | Кто вы — токен авторизации. Сервер проверяет: пускать или 401 | Необязательно (только для защищённых) |
| Заголовок | `Content-Type: application/json` | Тип тела запроса (если есть тело) | Обязательно, если есть тело |
| Заголовок | `Content-Length: 43` | Размер тела в байтах (если есть тело) | Обязательно, если есть тело |
| Заголовок | `Cookie: session=abc123` | Куки | Необязательно |
| **Пустая строка** | `\r\n` | Конец заголовков | Обязательно |
| **Тело** | `{"name":"Анна","email":"anna@mail.com"}` | Данные (у `GET` обычно пустое) | Необязательно |

**Структура HTTP-ответа:**
| Часть | Пример | Что означает | Обязательность |
|-------|--------|--------------|----------------|
| **Статус-строка** | `HTTP/1.1 200 OK` | Версия + код + текст | Обязательно |
| Заголовок | `Date: Sat, 18 Oct 2026 14:30:00 GMT` | Когда сгенерирован ответ | Обязательно по стандарту (часто опускают) |
| Заголовок | `Content-Type: application/json; charset=utf-8` | Что в теле | Обязательно, если есть тело |
| Заголовок | `Content-Length: 42` | Размер тела в байтах | Обязательно, если есть тело |
| Заголовок | `Cache-Control: max-age=3600` | Сколько кешировать | Необязательно |
| Заголовок | `Set-Cookie: session=abc123; HttpOnly; Path=/` | Поставь клиенту куку | Необязательно |
| Заголовок | `Location: /new-url` | Куда переехало (для 3xx) | Обязательно только для 3xx |
| **Пустая строка** | `\r\n` | Конец заголовков | Обязательно |
| **Тело** | `{"id":42,"name":"Анна"}` | Данные (может отсутствовать) | Необязательно |

## Метод

| Метод | Что делает | Тело? | Пример |
|-------|------------|-------|--------|
| `GET` | Получить данные | Нет | Забрать список пользователей из API |
| `POST` | Создать / отправить данные | Да | Отправить запрос к ML-модели на предсказание |
| `PUT` | Заменить объект целиком | Да | Обновить конфиг модели |
| `PATCH` | Обновить часть объекта | Да | Поменять только статус |
| `DELETE` | Удалить объект | Нет | Удалить модель из реестра |

## Путь + query

`Путь(куда я обращаюсь)?query` - если с путём всё понятно, то с параметрами запроса после знака `?` стоит слегка разобраться:
- `&` - разделитель между парами
- `=` - разделитель ключа и значения

То есть по сути все данные передаются как словарь, вот пример: `https://api.example.com/users?age=30&city=Москва&limit=10`

## Что такое куки?

Если в двух словах, то: "Кука — это небольшая порция данных, которую сервер просит клиента хранить и возвращать при каждом запросе; чаще всего это «номерок» сессии, избавляющий от повторной авторизации, но иногда — просто «записка» с настройками или ID для трекинга. В HTTP-запросе кука передаётся в заголовке Cookie в виде пар имя=значение, разделённых ; — например, `Cookie: session=abc123; theme=dark`. Клиент подставляет этот заголовок в каждый запрос к тому домену, который поставил куку, а сервер читает его и по значениям узнаёт клиента или его настройки"

## Коды

| Код | Обозначение | Что значит |
|-----|-------------|------------|
| **1xx — Информационные** | | |
| `100` | Continue | Сервер получил заголовки, можно слать тело |
| `101` | Switching Protocols | Переключение протокола (например, на WebSocket) |
| **2xx — Успех** | | |
| `200` | OK | Всё хорошо, данные в теле |
| `201` | Created | Ресурс создан (после `POST`) |
| `204` | No Content | Успех, но тела нет |
| **3xx — Перенаправление** | | |
| `301` | Moved Permanently | Переехало навсегда (SEO: вес переходит) |
| `302` | Found | Временный редирект |
| `304` | Not Modified | Не изменилось — бери из кеша |
| **4xx — Ошибка клиента** | | |
| `400` | Bad Request | Клиент прислал некорректный запрос |
| `401` | Unauthorized | Не авторизован (нет/протух токен) |
| `403` | Forbidden | Авторизован, но доступа нет |
| `404` | Not Found | Ресурс не найден |
| `405` | Method Not Allowed | Метод не поддерживается для этого URL |
| `409` | Conflict | Конфликт (например, дубликат при создании) |
| `422` | Unprocessable Entity | Синтаксис ок, но данные невалидны (часто в API) |
| `429` | Too Many Requests | Превышен rate limit |
| **5xx — Ошибка сервера** | | |
| `500` | Internal Server Error | Сервер упал (баг в коде) |
| `502` | Bad Gateway | Прокси не достучался до бэкенда |
| `503` | Service Unavailable | Сервер перегружен / на обслуживании |
| `504` | Gateway Timeout | Прокси не дождался ответа от бэкенда |